# SAFIRE ATR-42 calibration

**Aircraft class:** `hyplan.aircraft.SAFIRE_ATR42`

**Source:** Two archives: CEDA EUFAR (28 flights across 7 transnational-access projects) + AERIS EUREC4A 2020 (19 flights, native TAS).

CEDA EUFAR files lack TAS; reconstructed via wind triangle from position derivatives + wind components.  AERIS EUREC4A files ship native TAS and are used directly.

This notebook is a thin interactive companion to
[`calibrate.py`](./calibrate.py).  All loading, filtering, and
binning logic lives in that module so the standalone `python -m notebooks.calibration.SAFIRE_ATR42.calibrate`
invocation and this notebook produce identical numbers.  Edit the script
to change the recipe; re-run this notebook to inspect intermediate state
or regenerate the paste-ready constructor block for `_models.py`.


## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from the repo root so glob paths resolve.
sys.path.insert(0, str(Path("..").resolve().parents[1]))
sys.path.insert(0, str(Path("..").resolve()))
import notebooks.calibration.SAFIRE_ATR42.calibrate as cal
from _common import per_bin, tas_per_bin, schedule_pts

print("active-VS threshold:", cal.ACTIVE_VS_THR_FPM, "fpm")
print("duration filter:    ", cal.MIN_DUR_MIN, "..", cal.MAX_DUR_MIN, "min")
print("peak-altitude filter:", cal.MIN_PEAK_ALT_FT, "..", cal.MAX_PEAK_ALT_FT, "ft")
print("TAS schedule targets:", cal.TARGET_ALTS_FT)


## 2. Load every sortie

Reads all matching files, applies per-source loaders (with TAS
reconstruction where needed), filters out short / low-altitude / failed
sorties, and phase-labels by vertical-rate threshold.

In [ ]:
import os
os.chdir(Path("..").resolve().parents[1])  # repo root for glob patterns
sorties = cal.load_sorties()
print(f"\n{len(sorties)} sorties loaded")


## 3. Per-altitude-bin VS medians

Active-VS gate (per-aircraft threshold) on phase-labeled fixes,
binned in 5-kft steps with n>=30/bin.  Bins below the floor are
dropped as too thin to trust.

In [ ]:
climb_bins = per_bin(sorties, "climb", +1, cal.ACTIVE_VS_THR_FPM, n_min=30)
desc_bins = per_bin(sorties, "descent", -1, cal.ACTIVE_VS_THR_FPM, n_min=30)
print("CLIMB VS bins:")
print(climb_bins.to_string(index=False))
print("\nDESCENT VS bins:")
print(desc_bins.to_string(index=False))


## 4. TAS schedules per phase

Median TAS per altitude bin for cruise, climb, and descent fixes
(n>=200/bin).  Schedule breakpoints are picked at the per-aircraft
target altitudes.

In [ ]:
cruise_tas = tas_per_bin(sorties, ("cruise",), n_min=200)
climb_tas  = tas_per_bin(sorties, ("climb",),  n_min=200)
desc_tas   = tas_per_bin(sorties, ("descent",), n_min=200)
print("CRUISE TAS bins:")
print(cruise_tas.to_string(index=False))

cs   = schedule_pts(cruise_tas, cal.TARGET_ALTS_FT, n_min=200)
klms = schedule_pts(climb_tas,  cal.TARGET_ALTS_FT, n_min=200)
ds   = schedule_pts(desc_tas,   cal.TARGET_ALTS_FT, n_min=200)
print(f"\nCruise TAS schedule: {cs}")
print(f"Climb  TAS schedule: {klms}")
print(f"Desc.  TAS schedule: {ds}")


## 5. Climb-VS profile sanity plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
if not climb_bins.empty:
    ax.fill_betweenx(climb_bins["alt_bin_ft"], climb_bins["vs_p25"],
                      climb_bins["vs_p75"], alpha=0.25, label="IQR")
    ax.plot(climb_bins["vs_med"], climb_bins["alt_bin_ft"], "o-", label="climb median")
if not desc_bins.empty:
    ax.fill_betweenx(desc_bins["alt_bin_ft"], desc_bins["vs_p25"],
                      desc_bins["vs_p75"], alpha=0.25, color="C1", label="descent IQR")
    ax.plot(desc_bins["vs_med"], desc_bins["alt_bin_ft"], "s-", color="C1", label="descent median")
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("vertical rate (fpm)")
ax.set_ylabel("altitude bin (ft)")
ax.set_title("Per-bin active-VS medians (IQR shaded)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## 6. Approach speed, service ceiling, bank

* **Approach speed:** median TAS in the final 60 s of each sortie's airborne segment, where VS < -200 fpm.  AGL/MSL altitude alone doesn't isolate landing because many of these aircraft fly low-altitude science surveys at cruise speed.
* **Service ceiling:** operational p99 of per-sortie peak altitudes, NOT the certified airframe ceiling at MTOW.
* **Bank angle:** p90 of |roll| during turn-state fixes (>5° gate), capped at the AFM normal-ops 30° floor.

In [ ]:
final_rows = []
for a in sorties.values():
    if "tas_kt" not in a.columns or a.empty:
        continue
    t_end = a["timestamp"].iloc[-1]
    sub = a[a["timestamp"] >= t_end - pd.Timedelta(seconds=60)]
    sub = sub[sub["vertical_rate"] < -200]
    final_rows.append(sub[["tas_kt"]])
final = pd.concat(final_rows).dropna() if final_rows else pd.DataFrame()
approach_kt = float(final["tas_kt"].median()) if len(final) else float("nan")

peaks = [float(a["altitude"].max()) for a in sorties.values()]
ceiling = float(np.percentile(peaks, 99))

rolls = []
for a in sorties.values():
    if "roll_deg" in a.columns:
        r = a["roll_deg"].abs()
        rolls.append(r[r > 5.0])
roll_p90 = float(pd.concat(rolls).quantile(0.90)) if rolls else float("nan")

print(f"Approach TAS median:        {approach_kt:.0f} kt")
print(f"Service ceiling (op-p99):   {ceiling:.0f} ft")
if rolls:
    print(f"Bank angle p90 (|roll|>5°): {roll_p90:.1f}°")
else:
    print("Bank angle: not available; AFM default 30° floor applies")


## 7. Paste-ready constructor block

The block below mirrors what `python -m notebooks.calibration.SAFIRE_ATR42.calibrate` prints — keep them in
sync.  Paste into `hyplan/aircraft/_models.py` for the
`SAFIRE_ATR42` class.

In [ ]:
def _vs_pts(bins):
    return [(int(r["alt_bin_ft"]), int(round(r["vs_med"])))
            for _, r in bins.iterrows()]

print(f"# Calibrated against {len(sorties)} sorties.")
print(f"service_ceiling={int(round(ceiling/100)*100)} * ureg.feet,")
print(f"approach_speed={int(round(approach_kt))} * ureg.knot,")
print(f"climb_schedule=TasSchedule(points={klms!r}),")
print(f"cruise_schedule=TasSchedule(points={cs!r}),")
print(f"descent_schedule=TasSchedule(points={ds!r}),")
print(f"climb_profile=VerticalProfile(points={_vs_pts(climb_bins)!r}),")
print(f"descent_profile=VerticalProfile(points={_vs_pts(desc_bins)!r}),")
if rolls:
    print(f"max_bank_deg={max(30.0, round(roll_p90))},")
else:
    print("max_bank_deg=30.0,  # AFM default; no roll data in source")
